In [6]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import numpy as np
import scienceplots
import tifffile as tiff

from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia

plt.style.use('science')
plt.rcParams["figure.figsize"] = (3.5, 3.5 * ((5**0.5 - 1) / 2)) # 3.5
plt.rcParams["figure.dpi"] = 600
%matplotlib inline

import polars as pl
from polars import Expr, LazyFrame, DataFrame
import numpy as np
from pathlib import Path
from typing import Any
import tifffile
from typing import Dict
from typing import Callable


from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia

dp = DataProductEncyclopedia(
    data_products_path=Path(r"C:\Users\Joshu\OneDrive - Nexus365\AO33\Boulder_database\AO33\.data_products"))

In [7]:
outlined_img = cv2.imread("manual_data/manual_outlines.png")
outlines = outlined_img - np.mean(outlined_img, axis=2, dtype=np.float32)[:, :, None]
# cv2.imwrite("outlines.png", outlines)

In [8]:
# Convert to grayscale
gray = cv2.cvtColor(outlines.astype(np.uint8), cv2.COLOR_BGR2GRAY)

# Since rings are solid color, simple threshold works
_, binary = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)

# Find ALL contours including holes (important for rings)
contours, hierarchy = cv2.findContours(
    binary,
    cv2.RETR_TREE,          # keeps parent-child relationships
    cv2.CHAIN_APPROX_SIMPLE
)

print("Total contours:", len(contours))

Total contours: 301


In [9]:
outer_rings = []

for i, h in enumerate(hierarchy[0]):
    parent = h[3]
    if parent == -1:
        outer_rings.append(contours[i])

print("Number of rings:", len(outer_rings))

Number of rings: 130


In [ ]:
import cv2
import numpy as np
from typing import Tuple

# Copy original image for overlay
overlay = outlined_img.copy()

past_colour_codes : set[Tuple[int, int, int]] = {(0, 0, 0)} # Use (0, 0, 0) as the background

# Fill each ring with unique colour
for contour in outer_rings:
    while True:
        colour_code = (
            int(np.random.randint(0, 255, dtype=np.uint8)),
            int(np.random.randint(0, 255, dtype=np.uint8)),
            int(np.random.randint(0, 255, dtype=np.uint8))
        )

        if colour_code not in past_colour_codes:
            past_colour_codes.add(colour_code)
            break

    cv2.drawContours(overlay, [contour], -1, list(colour_code), thickness=cv2.FILLED)

shaded_boulder = overlay.copy()

# Preview the result
alpha = 0.5
cv2.imwrite(".plots/manual_detection_overlaid.png", cv2.addWeighted(overlay, alpha, outlined_img, 1 - alpha, 0))

True

In [13]:
df = dp.mask_atlas_combined.filter(pl.col("face") == "posx").collect()
df

lod_level,lod_code,face,i,j,row_id,uint8_reflectance,32bit_reflectance,positions_x,positions_y,positions_z
u8,str,str,u32,u32,u32,u8,f32,f32,f32,f32
1,"""A""","""posx""",0,577,19545,150,0.026051,-0.121165,-0.141084,-0.141084
1,"""A""","""posx""",0,589,19545,199,0.03446,-0.120867,-0.141213,-0.141212
1,"""A""","""posx""",0,605,19545,191,0.033173,-0.120426,-0.14134,-0.141346
4,"""AAAB""","""posx""",0,626,61029,129,0.022319,-0.119775,-0.141436,-0.141441
3,"""AAA""","""posx""",0,776,36995,33,0.005656,-0.11541,-0.142441,-0.142436
…,…,…,…,…,…,…,…,…,…,…
1,"""D""","""posx""",8168,7899,2907629,107,0.018463,0.13142,-0.141498,0.140671
1,"""D""","""posx""",8168,7922,2907629,192,0.033255,0.131975,-0.141241,0.14042
1,"""D""","""posx""",8168,8003,2907629,174,0.030146,0.133797,-0.140221,0.139399


In [ ]:
height = 8192
width = 8192

img = np.zeros((8192, 8192, 3), dtype=np.uint8)
i = df["i"].to_numpy()
j = df["j"].to_numpy()

full_atlas = dp.combined_atlas.filter(pl.col("face") == "posx").collect()
img[full_atlas["i"].to_numpy(), full_atlas["j"].to_numpy()] = full_atlas["uint8_reflectance"].to_numpy()[:, None]

img0 = np.zeros((height, width), dtype=np.uint8)
img1 = np.zeros((height, width), dtype=np.uint8)
img2 = np.zeros((height, width), dtype=np.uint8)

values = df["row_id"].to_numpy()

img[i, j, 0] = 0.5 * (values & 0xFF).astype(np.uint8) + 0.5 * img[i, j, 0]
img[i, j, 1] = 0.5 * ((values >> 8) & 0xFF).astype(np.uint8) + 0.5 * img[i, j, 1]
img[i, j, 2] = 0.5 * ((values >> 16) & 0xFF).astype(np.uint8) + 0.5 * img[i, j, 2]

cv2.imwrite(".plots/auto_detection_overlaid.png", img)

True